In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import(
    accuracy_score,classification_report,confusion_matrix,ConfusionMatrixDisplay
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ALPHA = 0.01
POPULATION_SIZE = 15
MAX_ITER = 40
K_MIN,K_MAX = 1,15
N_FEATURES = 13

print(f"✅Library Imported Successfull")
print(f"Random seed       : {RANDOM_SEED}")
print(f"Population Size   : {POPULATION_SIZE}")
print(f"Max iterations    : {MAX_ITER}")
print(f"K_Neighbour range : [{K_MIN}, {K_MAX}]")
print(f"Penalty alpha     : {ALPHA}")

In [ ]:
# Data Preprocessing

# X_train = Training data
# y_train = Output for Training data
# X_test = Testing data
# y_test = Output for Testing data

def load_data(test_size: float=0.3, random_state: int = RANDOM_SEED):
    wine = load_wine()
    X, y = wine.data, wine.target
    feature_names = list(wine.feature_names)
    class_names   = list(wine.target_names)

    X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=test_size,random_state=random_state,stratify=y)
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test) 

    return X_train, X_test, y_train, y_test, feature_names, class_names


X_train, X_test, y_train, y_test, FEATURE_NAMES, CLASS_NAMES = load_data(test_size=0.3)


print(f"Total Sample : 178")
print(f"Training Set : {X_train.shape[0]} Samples, {X_train.shape[1]} Features")
print(f"Testing Set  : {X_test.shape[0]} Sampeles")
print(f"Classes Names: {CLASS_NAMES}")
print(f"FEATURE_NAMES: {FEATURE_NAMES}")

In [ ]:
# Baseline KNN Model

# X_train = Input Data
# y_train = output data for input
def train_knn(X_train, y_train, k:int = 5, feature_mask: np.ndarray = None)-> KNeighborsClassifier:
    if feature_mask is not None:
        mask = feature_mask.astype(bool)
        if mask.sum()==0:
            mask[0]=True
        X_train = X_train[:,mask]
        
    classifier = KNeighborsClassifier(n_neighbors=int(k))
    classifier.fit(X_train,y_train)    # Learn on input and output
    return classifier



def evaluate_model(classifier: KNeighborsClassifier, X_test,y_test,feature_mask: np.ndarray = None)->float:
    # X_test = Testing data
    # y_test = output for Testing data

    if feature_mask is not None:
        mask = feature_mask.astype(bool)
        if mask.sum()==0:
            mask[0]=True
        X_test = X_test[:,mask]
    
    prediction = classifier.predict(X_test)
    prediction_accuracy = accuracy_score(y_test,prediction)
    return prediction_accuracy




def fitness_function(solution:np.ndarray, X_train, y_train):
    k = int(np.clip(round(solution[0]), K_MIN, K_MAX))
    feature_mask = (solution[1:] >= 0.5).astype(int)

    if feature_mask.sum() == 0:
        feature_mask[0] = 1

    X_selected = X_train[:, feature_mask.astype(bool)]

    model = KNeighborsClassifier(n_neighbors=k)

    scores = cross_val_score(model, X_selected, y_train, cv=5)
    accuracy = scores.mean()
    penalty = feature_mask.sum()/N_FEATURES

    return accuracy - penalty*ALPHA


baseline_model = train_knn(X_train,y_train,k=5,feature_mask=None)
baseline_accuracy = evaluate_model(baseline_model,X_test,y_test,feature_mask=None)

cross_validation_score = cross_val_score(KNeighborsClassifier(n_neighbors=5), X_train, y_train, cv=5)
# Mean of Accuracy on 5 Different validation

print("BASELINE KNN RESULT")
print(f"-"*50)
print(f"K (Neighbour)     : 5")
print(f"Feature Used      : All {N_FEATURES}")
print(f"Baseline Accuracy : {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")
print(f"5 Fold CV Accuracy: {cross_validation_score.mean():.4f} ± {cross_validation_score.std():.4f}")
print(f"-"*50)

In [ ]:
# Whale Optimization Model (From Scratch)

def whale_optimization(X_train,y_train,population_size: int=POPULATION_SIZE, max_iteration: int=MAX_ITER, random_state: int=RANDOM_SEED):

    rng = np.random.default_rng(random_state)
    dim = N_FEATURES+1

    lb = np.array([K_MIN] + [0.0]*N_FEATURES)
    ub = np.array([K_MAX] + [1.0]*N_FEATURES)

    position = lb + rng.random((population_size,dim)) * (ub-lb)

    initial_fitness = np.array([fitness_function(position[i],X_train,y_train) for i in range(population_size)])
    

    best_idx = np.argmax(initial_fitness)
    best_position = position[best_idx].copy()
    best_fitness_score = initial_fitness[best_idx]

    convergence = [best_fitness_score]

    # Whale Optimization Algorithm
    for t in range(max_iteration):
        a = 2.0 - t*(2.0/max_iteration)
        
        for i in range(population_size):
            r1,r2 = rng.random(), rng.random()
            A = 2.0 * a * r1 - a    # Movement Contoller(Exploration or exploiation)
            C = 2.0 * r2            # Distance Emphesis(How whales moves, same whale moves different distance becasue of it)
            b = 1.0                 # Spiral Shape Constant(How aggressive whale approach solution)
            l = rng.uniform(-1,1)   # Each whale takes different path in spiral to reach solution
            p = rng.random()        # p>0.5 then spiral, p<0.5 then encircling

            if p<0.5:
                if abs(A)<1:
                    # Encircling(Exploitation)
                    D = abs(C * best_position-position[i])
                    position[i] = best_position - A * D
                else: 
                    # Random Search(Exploration)
                    rnd_idx = rng.integers(0,population_size)
                    rnd_whale = position[rnd_idx]
                    D = abs(C * rnd_whale - position[i])
                    position[i] = rnd_whale - A * D
            else:
                # Bubble net spiral update
                D_leader = abs(best_position - position[i])
                position[i] = (D_leader * np.exp(b * l) * np.cos(2 * np.pi * l) + best_position)
            
            position[i] = np.clip(position[i],lb,ub)
            
            fit = fitness_function(position[i],X_train,y_train)                

            if fit > best_fitness_score:
                best_fitness_score = fit
                best_position = position[i].copy()

        
        convergence.append(best_fitness_score)

        if (t+1)%10==0 or t==0:
            print(f"WOA Iter: {t+1:3}/40  |  Best Fitness = {best_fitness_score:.4f}")
    
    return best_position,best_fitness_score,convergence


print(f"Running Whale Optimization Algorithm\n{'-'*50}")


woa_solution, woa_cv_accuracy, woa_convergence = whale_optimization(X_train,y_train,POPULATION_SIZE,MAX_ITER,RANDOM_SEED)

best_K = int(np.clip(round(woa_solution[0]),K_MIN,K_MAX))
best_feature_mask = (woa_solution[1:]>=0.5).astype(int)
best_features = [FEATURE_NAMES[i] for i,j in enumerate(best_feature_mask) if j==1]


woa_model = train_knn(X_train, y_train, best_K, best_feature_mask)
woa_test_accuracy = evaluate_model(woa_model,X_test,y_test,best_feature_mask)


print('-'*50)
print("✅ WOA COMPLETED!")
print('='*50)
print(f"WOA RESULTS\n{'-'*13}")
print(f"Best K-Neighbour      : {best_K}")
print(f"Total Feature Selected: {len(best_features)}/{N_FEATURES}")
print(f"Selected Features     : {best_features}\n")
print(f"WOA Test Accuracy     : {woa_test_accuracy:.4f} ({woa_test_accuracy*100.0:.2f}%)")
print(f"WOA CV Accuracy       : {woa_cv_accuracy:.4f} ({woa_cv_accuracy*100.0:.2f}%)")
